In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126 -q
!pip3 install scikit-image -q

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split

import copy
from skimage.feature import hog
from skimage.color import rgb2gray

import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import confusion_matrix
import numpy as np
import random
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


torch.manual_seed(1)
random.seed(1)
np.random.seed(1)

def get_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Implementación CNN modular

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, num_conv_layers, first_layer_filters, kernel_size, norm):
        super(ConvBlock, self).__init__()

        layers = []
        in_ch = in_channels

        for i in range(num_conv_layers):
            out_ch = first_layer_filters * (2 ** i)

            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size, padding=1))

            if norm:
                layers.append(nn.BatchNorm2d(out_ch))

            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        self.block = nn.Sequential(*layers)
        out_ch = first_layer_filters * (2 ** (num_conv_layers - 1))
        self.last_filters = out_ch

    def forward(self, x):
        return self.block(x)


class FCBlock(nn.Module):
    def __init__(self, in_features, hidden_mlp_layers, output_neurons):
        super(FCBlock, self).__init__()

        layers = []
        for hidden_size in hidden_mlp_layers:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size

        layers.append(nn.Linear(in_features, output_neurons))
        # Sin Softmax (CrossEntropyLoss lo incluye internamente)

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [ ]:
class ModuleCNN(nn.Module):
    def __init__(self, init_h, init_w, in_channels, num_conv_layers, first_layer_filters,
                 kernel_size, norm, hidden_mlp_layers, output_neurons):
        super(ModuleCNN, self).__init__()

        self.conv = ConvBlock(in_channels, num_conv_layers, first_layer_filters, kernel_size, norm)

        # Cálculo a mano del tamaño de la salida del bloque convolucional (alternativa al dummy)
        h, w = init_h, init_w
        for _ in range(num_conv_layers):
            # conv
            h = (h+2) - kernel_size + 1 # padding de 1, stride de 1
            w = (w+2) - kernel_size + 1
            # pooling
            h = h //2
            w = w //2
        flat_size = self.conv.last_filters * h * w

        self.fc = FCBlock(flat_size, hidden_mlp_layers, output_neurons)

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
cnn = ModuleCNN(init_h=160, init_w=160, in_channels=3, num_conv_layers=5, first_layer_filters=8,
                kernel_size=3, norm=True, hidden_mlp_layers=[128], output_neurons=10)
print(f"O modelo ten {get_parameters(cnn):,} parámetros")

### DATASETS

In [ ]:
# Descargar conjunto Imaginette
datasets.Imagenette(root="data/", split="train", size="160px", download=True)

# dividir en train y test
train_data = datasets.Imagenette(root="data/", split="train", size="160px", download=False, transform=transforms.Compose([transforms.Resize((160, 160)), transforms.ToTensor()]))
val_data = datasets.Imagenette(root="data/", split="val", size="160px", download=False, transform=transforms.Compose([transforms.Resize((160, 160)), transforms.ToTensor()]))

In [ ]:
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

full_train_data = datasets.Imagenette(root="data/", split="train", size="160px", download=False, transform=transform)
test_data       = datasets.Imagenette(root="data/", split="val",   size="160px", download=False, transform=transform)
print(f"Total de imaxes de adestramento: {len(full_train_data)}")
print(f"Total de imaxes de test: {len(test_data)}")
train_size = int(0.85 * len(full_train_data))
val_size   = len(full_train_data) - train_size
train_dataset, val_dataset = random_split(full_train_data, [train_size, val_size])

batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_dataloader  = DataLoader(test_data,     batch_size=batch_size, shuffle=False)

print(f"Número de lotes de adestramento: {len(train_dataloader)}")
print(f"Número de lotes de validación: {len(val_dataloader)}")
print(f"Número de lotes de test: {len(test_dataloader)}")
for batch_imgs, batch_labels in train_dataloader:
    print(f"Tamaño das imaxes: {batch_imgs.size()} - Etiquetas: {batch_labels}")
    break # Paramos o bucle, xa que só queremos ver un lote

In [ ]:
for batch_imgs, batch_labels in train_dataloader:
    print(f"Tamaño das imaxes: {batch_imgs.size()} - Etiquetas: {batch_labels}")
    break

### Ejecución del modelo

#### Posibles hiperparámetros:
- learning rate
- optimizer
- n capas
- n neuronas
- n filtros
- n epochs
- tam lotes
- ¿padding,tam kernel en convolución?

In [ ]:
# ── FUNCIONES DE TRAIN / EVAL ────────────────────────────────────────────────
def train(dataloader, model, loss_fn, optimizer):
    num_images = len(dataloader.dataset)
    model.train()
    n_batch = 0
    for batch_imgs, batch_labels in dataloader:
        batch_predicted_probabilities = model(batch_imgs)
        loss = loss_fn(batch_predicted_probabilities, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if n_batch % 100 == 0:
            imgs_processed = n_batch * len(batch_imgs)
            print(f"  perda: {loss.item():>7f}  [{imgs_processed:>5d}/{num_images:>5d}]")
        n_batch += 1

    return loss  # última pérdida del epoch (igual que el tutorial)


def evaluate(dataloader, model, loss_fn):
    """Devuelve (loss_media, predicted_classes, true_classes)"""
    num_images = len(dataloader.dataset)
    model.eval()
    total_loss, correct = 0, 0
    predicted_classes, true_classes = [], []

    with torch.no_grad():
        for batch_imgs, batch_labels in dataloader:
            probs = model(batch_imgs)
            total_loss += loss_fn(probs, batch_labels).item()
            preds = probs.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
            predicted_classes.extend(preds.tolist())
            true_classes.extend(batch_labels.tolist())

    avg_loss = total_loss / num_images
    accuracy = correct / num_images
    print(f"  Perda media: {avg_loss:>8f}  Accuracy: {accuracy*100:>0.1f}%")
    return avg_loss, predicted_classes, true_classes

In [ ]:
def train_with_early_stopping(model, train_dataloader, val_dataloader, 
                               epochs=20, patience=3, lr=1e-3):
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss    = float('inf')
    best_model_state = None
    current_patience = 0
    train_losses, val_losses = [], []

    for t in range(epochs):
        print(f"\nEpoch {t+1}/{epochs}\n{'-'*35}")
        train_loss = train(train_dataloader, model, loss_fn, optimizer)
        train_losses.append(train_loss.item())

        val_loss, _, _ = evaluate(val_dataloader, model, loss_fn)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            current_patience = 0
            print("  ✓ Mellor modelo gardado")
        else:
            current_patience += 1
            print(f"  Paciencia: {current_patience}/{patience}")
            if current_patience >= patience:
                print("  Early stopping activado")
                break

    model.load_state_dict(best_model_state)
    return model, best_val_loss, train_losses, val_losses

In [ ]:
configs = [
    # hay que hacer el grid de parámetros
]

best_global_val_loss = float('inf')
best_global_model    = None
results = []

for cfg in configs:
    print(f"\n{'='*50}")
    print(f"Config: {cfg}")

    model = ModuleCNN(
        # parámetros fijos y cfg
    )
    print(f"Parámetros: {get_parameters(model):,}")

    model, best_val_loss, train_losses, val_losses = train_with_early_stopping(
        model, train_dataloader, val_dataloader, epochs=20, patience=3
    )

    # Evaluar en test
    loss_fn = nn.CrossEntropyLoss()
    _, test_predicted, test_true = evaluate(test_dataloader, model, loss_fn)
    acc = accuracy_score(test_true, test_predicted)
    precision = precision_score(test_true, test_predicted, average='macro')
    recall = recall_score(test_true, test_predicted, average='macro')
    f1  = f1_score(test_true, test_predicted, average='macro')

    results.append({**cfg, "params": get_parameters(model),
                    "val_loss": best_val_loss, "accuracy": acc, "precision": precision,
                    "recall": recall, "f1": f1, "train_losses": train_losses, "val_losses": val_losses})

    # Actualizar mejor modelo global (comparando por val_loss)
    if best_val_loss < best_global_val_loss:
        best_global_val_loss = best_val_loss
        best_global_model    = copy.deepcopy(model.state_dict())
        print(f"  ★ Novo mellor modelo global (val_loss={best_val_loss:.6f})")

# Mostrar tabla resumen
results_df = pd.DataFrame(results).drop(columns=["train_losses", "val_losses"])
print(results_df.to_string(index=False))

# Cargar el mejor modelo global para uso posterior
best_cnn = ModuleCNN(...)  # con la config ganadora
best_cnn.load_state_dict(best_global_model)